## Load the Dataset

In [7]:
from pyspark.sql import SparkSession
import pandas as pd
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook"

# Initialize Spark Session
spark = SparkSession.builder.appName("LightcastData").getOrCreate()

# Load Data
df = spark.read.option("header", "true").option("inferSchema", "true").option("multiLine","true").option("escape", "\"").csv("data/lightcast_job_postings.csv")
df.createOrReplaceTempView("job_postings")
# Show Schema and Sample Data
# print("---This is Diagnostic check, No need to print it in the final doc---")

# df.printSchema() # comment this line when rendering the submission
# df.show(5)

26/06/18 22:17:00 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


## Feature Engineering

In [11]:
target_df=spark.sql("""
SELECT
	salary,
	min_years_experience,
	min_edulevels,
	lot_v6_specialized_occupation_name AS occupation
FROM job_postings
GROUP BY 
	salary,
 	min_years_experience,
	min_edulevels,
	occupation
""")
target_df.show()
target_df.createOrReplaceTempView("target_df")

+------+--------------------+-------------+--------------------+
|salary|min_years_experience|min_edulevels|          occupation|
+------+--------------------+-------------+--------------------+
| 98650|                   5|            2|Business Analyst ...|
| 55570|                   1|            2| SAP Analyst / Admin|
| 87037|                   1|            1|        Data Analyst|
|187200|                   7|            2|Enterprise Architect|
|160000|                   5|            2|General ERP Analy...|
|133765|                   5|            2|Oracle Consultant...|
|109200|                   8|            2|        Data Analyst|
|114165|                   6|            2|Oracle Consultant...|
|132500|                   5|            2|Enterprise Architect|
|153920|                   7|           99|        Data Analyst|
|122720|                   3|            0| SAP Analyst / Admin|
|101920|                   4|            2|Oracle Consultant...|
|106000|                 

In [14]:
target = 'salary'
key_features = ['min_years_experience', 'min_edulevels', 'occupation']
df_clean = target_df.dropna(subset=[target]+key_features)
df_clean.toPandas().head(5).style.hide(axis="index")

salary,min_years_experience,min_edulevels,occupation
98650,5,2,Business Analyst (General)
55570,1,2,SAP Analyst / Admin
87037,1,1,Data Analyst
187200,7,2,Enterprise Architect
160000,5,2,General ERP Analyst / Consultant


In [15]:
from pyspark.ml.feature import StringIndexer

occupation_indexer = StringIndexer(inputCol='occupation', outputCol='occupationIndex')
indexed = occupation_indexer.fit(target_df).transform(target_df)
column_list = ['occupationIndex']
for column_name in column_list:
    indexed = indexed.withColumn(column_name, col(column_name).cast(DecimalType(18, 2)))
indexed.select("ocupation", "ocupationIndex"
               ).toPandas().head(5).style.hide(axis="index")


NameError: name 'col' is not defined

## Train/Test Split

## Linear Regression

## Generalized Linear Regression Summary

## Diagnostic Plot

## Evaluation